# Scribble Evaluation Notebook
Load a scribble from Google Drive, generate conditioned photos,
compute MMD vs 5-class target distribution, and classify via 5-way cosine softmax.

## 1. Setup

In [ ]:
import os, sys, json, gc
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from IPython.display import display
import wandb
from sklearn.decomposition import PCA
import matplotlib.cm as cm
import random

# Make src/ importable — works from notebooks/ or repo root
_NB_DIR  = Path().resolve()
_REPO    = _NB_DIR.parent if (_NB_DIR.parent / 'src').exists() else _NB_DIR
_SRC_DIR = str(_REPO / 'src')
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

GLOBAL_SEED = 999

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'[Seed] All random seeds set to {seed}')

set_global_seed(GLOBAL_SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


## 2. Config & Load Scribble

In [ ]:

CONTROLNET_SCALE = 0.5
N_EVAL=75
N_TARGETS        = 300
EVAL_PROMPT      = 'a superrealistic professional photograph of'

# ── 5-class target distribution
TARGET_PROMPTS = [
    ('Woman',            'superrealistic portrait photograph of a woman, extremely feminine features, studio lighting',                                                                          0.25),
    ('Woman with masculine features',      'a superrealistic portrait photograph of a woman with masculine features, heavy brow ridge, studio lighting',                    0.25),
    ('Man with feminine features',  'a superrealistic portrait photograph of a man with extremely feminine feminine features, soft delicate face, high cheekbones, studio lighting',       0.25),
    ('Man',              'a superrealistic portrait photograph of a man, extremely masculine features, studio lighting',                                                                             0.25),
]
assert abs(sum(r for _, _, r in TARGET_PROMPTS) - 1.0) < 1e-6

CLASS_LABELS  = [label  for label, _, _  in TARGET_PROMPTS]
CLASS_PROMPTS = [prompt for _, prompt, _ in TARGET_PROMPTS]
CLASS_FRACS   = [frac   for _, _, frac   in TARGET_PROMPTS]
CLASS_COLORS  = ['crimson', 'orchid', 'slategray', 'steelblue', 'royalblue']

# n images per class in the target
n_per_class = [int(N_TARGETS * f) for f in CLASS_FRACS]

print('Class fractions:')
for label, n in zip(CLASS_LABELS, n_per_class):
    print(f'  {label:<20} n={n}')

## 3. Load Models

In [ ]:
from models     import load_models
from clip_utils import load_clip_model

architect, sprinter        = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('Models loaded.')

## 4. Helpers

In [ ]:
from generation    import generate_and_store_cs
from clip_utils    import encode_images_clip
from visualization import plot_row
from metrics       import compute_mmd

def pil_to_tensor(pil_list):
    return torch.cat(
        [TF.to_tensor(img).unsqueeze(0) for img in pil_list], dim=0
    ).to(next(clip_model.parameters()).device)

def generate_eval_photos(scribble_pil, n=N_EVAL, seed=None):
    sprinter.vae.to(dtype=torch.float16)
    generator = None
    if seed is not None:
        generator = torch.Generator(device=sprinter.device).manual_seed(seed)
    photos = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            result = sprinter(
                prompt=[EVAL_PROMPT] * bs,
                image=[scribble_pil] * bs,
                num_inference_steps=2,
                guidance_scale=0.0,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                output_type='pil',
                generator=generator,
            )
            photos.extend(result.images)
    sprinter.vae.to(dtype=torch.float32)
    return photos

def generate_and_store_cs(pipe, prompt, cond_pil, num_samples, batch_size=2, cn_scale=0.5, seed=None):
    original_vae_dtype = pipe.vae.dtype
    pipe.vae.to(dtype=torch.float16)
    all_images, all_lats = [], []

    def latents_callback(p, step_index, timestep, cb_kwargs):
        if step_index == p.num_timesteps - 1:
            p._current_latents = cb_kwargs['latents'].detach().cpu().numpy()
        return cb_kwargs

    generator = None
    if seed is not None:
        generator = torch.Generator(device=pipe.device).manual_seed(seed)
    for i in range(0, num_samples, batch_size):
        curr = min(batch_size, num_samples - i)
        result = pipe(
            prompt=[prompt] * curr,
            image=[cond_pil] * curr,
            num_inference_steps=2,
            guidance_scale=0.0,
            controlnet_conditioning_scale=cn_scale,
            callback_on_step_end=latents_callback,
            generator=generator,
        )
        all_images.extend(result.images)
        all_lats.append(pipe._current_latents.reshape(curr, -1))
        print(f'  Progress: {len(all_images)}/{num_samples}', end='\r')
    print()
    pipe.vae.to(dtype=original_vae_dtype)
    return all_images, np.vstack(all_lats)


def encode_text_prompts(prompts):
    clip_model.to(device)
    inputs = clip_processor(
        text=prompts,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        # use pooler_output directly from the output we already saw
        outputs = clip_model.text_model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        text_embs = outputs.pooler_output  # [N, D] — this is a plain tensor

    # project through the text projection layer (same as get_text_features does internally)
    text_embs = clip_model.text_projection(text_embs)

    clip_model.to('cpu')
    text_embs = text_embs / text_embs.norm(dim=-1, keepdim=True)
    return text_embs  # [N, D]




def classify_multinomial(image_embs, text_embs):
    """
    5-way cosine softmax classification.
    image_embs: [N, D] (already L2-normalised from encode_images_clip)
    text_embs:  [C, D] (L2-normalised)
    Returns:
        labels  : list of predicted class indices  [N]
        probs   : np.array [N, C]  softmax probabilities
        proportions: np.array [C]  fraction predicted per class
    """
    # ensure both on same device
    image_embs = image_embs.to(device)
    text_embs  = text_embs.to(device)

    image_embs_n = F.normalize(image_embs.float(), dim=-1)
    logits = image_embs_n @ text_embs.T.float()             # [N, C]
    probs  = torch.softmax(logits * 100, dim=-1).cpu().numpy()
    labels = probs.argmax(axis=1).tolist()
    proportions = np.bincount(labels, minlength=len(CLASS_LABELS)) / len(labels)
    return labels, probs, proportions


def multinomial_ci_normal(counts, n_total, z=1.96):
    """Normal-approximation 95% CI for each class proportion."""
    p_hats = counts / n_total
    ses    = np.sqrt(p_hats * (1 - p_hats) / n_total)
    return p_hats, p_hats - z * ses, p_hats + z * ses


print('Helpers ready.')

## 5. Load Best Run Scribble

In [ ]:
## 5. Load Best Run Scribble

_REPO_ROOT = Path(_SRC_DIR).parent
run_dir    = _REPO_ROOT / 'experiments' / 'GenderInterpolation'

scribble_pil_1  = Image.open(run_dir / 'scribble_mlgdd.png')
source_scribble = Image.open(run_dir / 'scribble_source.png')
source_img      = Image.open(run_dir / 'source_portrait.png')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [source_img, source_scribble, scribble_pil_1],
    ['Source portrait', 'Source scribble (input)', 'MLGD-F scribble'],
):
    ax.imshow(img); ax.axis('off'); ax.set_title(title)
plt.tight_layout(); display(fig); plt.close()

scribble_pil = scribble_pil_1
print(f'Loaded scribble from {run_dir}')


## 6. Build Target Distribution

In [ ]:
print(f'Building target distribution ({N_TARGETS} images, 5 classes)...')
all_target_images = []
with torch.no_grad():
    for i, (label, prompt, frac) in enumerate(TARGET_PROMPTS):
        n_i = n_per_class[i]
        print(f'  [{label}] {n_i} images...')
        imgs, _ = generate_and_store_cs(sprinter, prompt, scribble_pil_1, n_i,
                                        batch_size=2, cn_scale=CONTROLNET_SCALE,
                                        seed=GLOBAL_SEED + i * 1000)
        all_target_images.extend(imgs)
        plot_row(imgs, f'Target {label} ({n_i})', count=min(8, n_i))

with torch.no_grad():
    all_clip_embeddings = encode_images_clip(
        pil_to_tensor(all_target_images), clip_model, clip_processor
    )
print(f'Target CLIP embeddings: {all_clip_embeddings.shape}')

## 7. Generate Eval Photos from Scribble

In [ ]:
eval_photos_1 = generate_eval_photos(scribble_pil_1, n=N_EVAL, seed=GLOBAL_SEED)
plot_row(eval_photos_1, 'MLGD-D', count=min(10, len(eval_photos_1)))

In [ ]:
## 8. Fit PCA & Sort Eval Photos
from sklearn.decomposition import PCA

# Encode eval photos
with torch.no_grad():
    eval_embs = encode_images_clip(
        pil_to_tensor(eval_photos_1), clip_model, clip_processor
    ).cpu().numpy()

# Extract woman and man embeddings from target distribution
offsets = np.cumsum([0] + n_per_class)
woman_embs = all_clip_embeddings[offsets[0]:offsets[1]].cpu().numpy()
man_embs   = all_clip_embeddings[offsets[3]:offsets[4]].cpu().numpy()

# Fit PCA on Woman + Man only
pca = PCA(n_components=2)
pca.fit(np.vstack([woman_embs, man_embs]))

eval_2d = pca.transform(eval_embs)



In [ ]:
# Orient so feminine=left, masculine=right
flip = -1 if man_embs.mean(0) @ pca.components_[0] < woman_embs.mean(0) @ pca.components_[0] else 1
pc1_scores = eval_2d[:, 0] * flip

sorted_idx    = np.argsort(pc1_scores)
sample_photos = eval_photos_1

In [ ]:
n = len(sample_photos)
N_PHOTOS=16
picks = [sorted_idx[i * (n // N_PHOTOS)] for i in range(N_PHOTOS)]

fig, axes = plt.subplots(2, N_PHOTOS//2, figsize=(20, 6))
for ax, idx in zip(axes.flatten(), picks):
    ax.imshow(sample_photos[idx])
    ax.axis('off')

axes[0, 0].set_title('← Woman', fontsize=10, color='crimson')
axes[0, 6].set_title('Man →',   fontsize=10, color='steelblue')
fig.suptitle('Gender axis — 14 photos from feminine to masculine', fontsize=12)
plt.tight_layout()
display(fig)
plt.close()

In [ ]:

# ── 1. Generate eval photos ONCE ──────────────────────────────────────────
print(f'Generating {N_TARGETS} eval photos from MLGD-D scribble...')
eval_photos = generate_eval_photos(scribble_pil_1, n=N_TARGETS, seed=GLOBAL_SEED)

# ── 2. Encode eval photos ONCE ────────────────────────────────────────────
with torch.no_grad():
    eval_embs = encode_images_clip(
        pil_to_tensor(eval_photos), clip_model, clip_processor
    ).cpu().numpy()

# ── 3. Pull out class embeddings ──────────────────────────────────────────
offsets = np.cumsum([0] + n_per_class)
woman_embs      = all_clip_embeddings[offsets[0]:offsets[1]].cpu().numpy()
woman_masc_embs = all_clip_embeddings[offsets[1]:offsets[2]].cpu().numpy()
man_fem_embs    = all_clip_embeddings[offsets[2]:offsets[3]].cpu().numpy()
man_embs        = all_clip_embeddings[offsets[3]:offsets[4]].cpu().numpy()

# ── 4. Fit PCA on Woman + Man ONLY ────────────────────────────────────────
pca = PCA(n_components=2)
pca.fit(np.vstack([woman_embs, man_embs]))

# ── 5. Project everything ONCE ────────────────────────────────────────────
woman_2d      = pca.transform(woman_embs)
woman_masc_2d = pca.transform(woman_masc_embs)
man_fem_2d    = pca.transform(man_fem_embs)
man_2d        = pca.transform(man_embs)
eval_2d       = pca.transform(eval_embs)

# ── 6. Orient so Man = positive on PC1 ───────────────────────────────────
flip = -1 if man_2d[:, 0].mean() < woman_2d[:, 0].mean() else 1

all_pc1 = np.concatenate([
    woman_2d[:, 0], woman_masc_2d[:, 0],
    man_fem_2d[:, 0], man_2d[:, 0], eval_2d[:, 0]
]) * flip
vmin, vmax = all_pc1.min(), all_pc1.max()

def score(coords):
    return coords[:, 0] * flip


In [ ]:

cmap = cm.RdBu_r
bg_size = 120

fig, ax = plt.subplots(figsize=(16, 5.4))

# Target distribution — circles, colored by gender score
for embs_2d in [woman_2d, woman_masc_2d, man_fem_2d, man_2d]:
    ax.scatter(embs_2d[:, 0], embs_2d[:, 1],
               c=score(embs_2d), cmap=cmap, vmin=vmin, vmax=vmax,
               s=bg_size, marker='o', alpha=0.6,
               edgecolors='black', linewidths=0.5)

# MLGD-D eval photos — stars, colored by gender score
ax.scatter(eval_2d[:, 0], eval_2d[:, 1],
           c='gold', s=int(bg_size*1.5), marker='o', alpha=1.0,
           edgecolors='black', linewidths=0.8, zorder=10)

ax.set_xlabel('PC1', fontsize=26, labelpad=15)
ax.set_ylabel('PC2', fontsize=26, labelpad=15)
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.tick_params(axis='both', which='both', length=0)
ax.grid(True, linestyle='--', alpha=0.3, color='gray')
ax.set_axisbelow(True)

plt.tight_layout()
display(fig)
plt.close()